# Expanded NHANES elevated-HbA1c classification experiment

This standalone research notebook evaluates whether verified diet, background,
and contextual variables add predictive information beyond WellPath-style KPIs.

**Outcome:** `elevated_hba1c = 1` when measured HbA1c is at least 5.7%, otherwise
0. This is a cross-sectional classification target—not a diagnosis of diabetes,
not a prediction of future diabetes, and not a clinically validated tool.

The analysis uses only the files in `data/raw/nhanes/`. It never invents missing
variables and never uses WellPath application users for model performance.

## Design and safeguards

- One participant row per `SEQN`; joins are validated one-to-one.
- Stratified 80/20 train/test splits with `random_state=42`.
- Five-fold stratified cross-validation occurs only inside training data.
- Imputation, scaling, and one-hot encoding are fitted inside sklearn pipelines.
- Direct target fields, glucose, diagnoses, medications, and insulin are forbidden.
- The strict complete-case comparison requires every experiment predictor.
- A separate shared cohort compares KPI + demographics, diet, the combined
  medical/family-risk proxy, and the shared full model without unrelated groups.
- Available-case models require at least 60% of their own predictors to be
  observed before training-only imputation.
- Direct incremental comparisons use identical held-out participants.
- Confidence intervals use 1,000 fixed-seed bootstrap resamples.
- Diet recalls are one or two 24-hour snapshots, not perfect usual-diet measures.

Official wording was checked against the CDC/NCHS 2017–2018 component codebooks
for DIQ_J, DBQ_J, DR1TOT_J, and DR2TOT_J.

In [1]:
from pathlib import Path
import sys
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.expanded_analysis import run_expanded_analysis

results = run_expanded_analysis(PROJECT_ROOT)
summary = results["summary"]
print("Expanded analysis completed.")
print(summary)

Expanded analysis completed.
{'eligible_participants': 6045, 'positive_participants': 2386, 'negative_participants': 3659, 'diet_variables_found': ['diet_calories', 'diet_carbohydrate_g', 'diet_total_sugar_g', 'diet_fiber_g', 'diet_protein_g', 'diet_total_fat_g', 'diet_saturated_fat_g', 'diet_sodium_mg', 'diet_potassium_mg', 'diet_cholesterol_mg', 'diet_alcohol_g', 'meals_away_from_home_7d', 'fast_food_meals_7d', 'ready_to_eat_meals_30d', 'frozen_meals_30d', 'self_rated_diet_quality'], 'family_history_status': 'Combined medical/family-risk proxy found', 'smoking_variables_found': [], 'food_security_variables_found': [], 'healthcare_access_variables_found': [], 'models_successfully_trained': 35, 'available_case_min_predictor_fraction': 0.6, 'complete_case_sample_size': 2713, 'shared_comparison_sample_size': 2713, 'available_case_full_model_sample_size': 5671, 'best_complete_case_model': 'Model 14: Full with BMI only', 'best_complete_case_estimator': 'Regularized logistic regression', 'b

## 1. NHANES file inventory

In [2]:
inventory = results["inventory"]
display(inventory.groupby("filename", as_index=False).agg(
    rows=("number_of_rows", "first"),
    columns=("number_of_columns", "first"),
    seqn_exists=("seqn_exists", "first"),
    status=("status", "first"),
))
print(f"Full inventory rows (one per source column): {len(inventory):,}")
display(inventory.head(20))

,filename,rows,columns,seqn_exists,status
0,BMX_J.xpt,8704,21,True,loaded
1,BPX_J.xpt,8704,21,True,loaded
2,DBQ_J.xpt,9254,46,True,loaded
3,DEMO_J.xpt,9254,46,True,loaded
4,DIQ_J.xpt,8897,54,True,loaded
5,DR1TOT_J.xpt,8704,168,True,loaded
6,DR2TOT_J.xpt,8704,85,True,loaded
7,GHB_J.xpt,6401,2,True,loaded
8,OCQ_J.xpt,6161,10,True,loaded
9,PAQ_J.xpt,5856,17,True,loaded


Full inventory rows (one per source column): 481


,filename,number_of_rows,number_of_columns,seqn_exists,all_column_names,column_name,column_dtype,status,message
0,BMX_J.xpt,8704,21,True,SEQN; BMDSTATS; BMXWT; BMIWT; BMXRECUM; BMIREC...,SEQN,float64,loaded,
1,BMX_J.xpt,8704,21,True,SEQN; BMDSTATS; BMXWT; BMIWT; BMXRECUM; BMIREC...,BMDSTATS,float64,loaded,
2,BMX_J.xpt,8704,21,True,SEQN; BMDSTATS; BMXWT; BMIWT; BMXRECUM; BMIREC...,BMXWT,float64,loaded,
3,BMX_J.xpt,8704,21,True,SEQN; BMDSTATS; BMXWT; BMIWT; BMXRECUM; BMIREC...,BMIWT,float64,loaded,
4,BMX_J.xpt,8704,21,True,SEQN; BMDSTATS; BMXWT; BMIWT; BMXRECUM; BMIREC...,BMXRECUM,float64,loaded,
5,BMX_J.xpt,8704,21,True,SEQN; BMDSTATS; BMXWT; BMIWT; BMXRECUM; BMIREC...,BMIRECUM,float64,loaded,
6,BMX_J.xpt,8704,21,True,SEQN; BMDSTATS; BMXWT; BMIWT; BMXRECUM; BMIREC...,BMXHEAD,float64,loaded,
7,BMX_J.xpt,8704,21,True,SEQN; BMDSTATS; BMXWT; BMIWT; BMXRECUM; BMIREC...,BMIHEAD,float64,loaded,
8,BMX_J.xpt,8704,21,True,SEQN; BMDSTATS; BMXWT; BMIWT; BMXRECUM; BMIREC...,BMXHT,float64,loaded,
9,BMX_J.xpt,8704,21,True,SEQN; BMDSTATS; BMXWT; BMIWT; BMXRECUM; BMIREC...,BMIHT,float64,loaded,


The full machine-readable inventory includes every column name and dtype
and is saved as `outputs/tables/nhanes_file_inventory.csv`.

## 2. Variable discovery

In [3]:
display(results["discovery"])
print("Verified feature groups:")
for group, fields in results["groups"].items():
    print(f"- {group}: {fields if fields else 'none available'}")

,requested_concept,participant_feature,source_day,matching_source_file,matching_variable_code,variable_label,response_coding,missing_value_codes,decision,concept_available,notes
0,Calories,diet_calories,day 1,DR1TOT_J,DR1TKCAL,Energy (kcal),Continuous daily intake; participant feature a...,. = missing,usable,True,Source column loaded. One or two 24-hour recal...
1,Calories,diet_calories,day 2,DR2TOT_J,DR2TKCAL,Energy (kcal),Continuous daily intake; participant feature a...,. = missing,usable,True,Source column loaded. One or two 24-hour recal...
2,Carbohydrate,diet_carbohydrate_g,day 1,DR1TOT_J,DR1TCARB,Carbohydrate (gm),Continuous daily intake; participant feature a...,. = missing,usable,True,Source column loaded. One or two 24-hour recal...
3,Carbohydrate,diet_carbohydrate_g,day 2,DR2TOT_J,DR2TCARB,Carbohydrate (gm),Continuous daily intake; participant feature a...,. = missing,usable,True,Source column loaded. One or two 24-hour recal...
4,Total Sugar,diet_total_sugar_g,day 1,DR1TOT_J,DR1TSUGR,Total sugars (gm),Continuous daily intake; participant feature a...,. = missing,usable,True,Source column loaded. One or two 24-hour recal...
5,Total Sugar,diet_total_sugar_g,day 2,DR2TOT_J,DR2TSUGR,Total sugars (gm),Continuous daily intake; participant feature a...,. = missing,usable,True,Source column loaded. One or two 24-hour recal...
6,Fiber,diet_fiber_g,day 1,DR1TOT_J,DR1TFIBE,Dietary fiber (gm),Continuous daily intake; participant feature a...,. = missing,usable,True,Source column loaded. One or two 24-hour recal...
7,Fiber,diet_fiber_g,day 2,DR2TOT_J,DR2TFIBE,Dietary fiber (gm),Continuous daily intake; participant feature a...,. = missing,usable,True,Source column loaded. One or two 24-hour recal...
8,Protein,diet_protein_g,day 1,DR1TOT_J,DR1TPROT,Protein (gm),Continuous daily intake; participant feature a...,. = missing,usable,True,Source column loaded. One or two 24-hour recal...
9,Protein,diet_protein_g,day 2,DR2TOT_J,DR2TPROT,Protein (gm),Continuous daily intake; participant feature a...,. = missing,usable,True,Source column loaded. One or two 24-hour recal...


Verified feature groups:
- core_kpis: ['sleep_hours', 'resting_heart_rate', 'systolic_bp', 'diastolic_bp', 'bmi', 'weight_kg', 'vigorous_activity', 'moderate_activity', 'sedentary_minutes']
- demographics: ['age', 'sex', 'income_context']
- diet_nutrients: ['diet_calories', 'diet_carbohydrate_g', 'diet_total_sugar_g', 'diet_fiber_g', 'diet_protein_g', 'diet_total_fat_g', 'diet_saturated_fat_g', 'diet_sodium_mg', 'diet_potassium_mg', 'diet_cholesterol_mg', 'diet_alcohol_g']
- diet_behaviours: ['meals_away_from_home_7d', 'fast_food_meals_7d', 'ready_to_eat_meals_30d', 'frozen_meals_30d', 'self_rated_diet_quality']
- family_history: none available
- medical_family_risk_proxy: ['medical_or_family_diabetes_risk']
- smoking: none available
- food_security: none available
- healthcare_access: none available


## 3. Family-history wording review

In [4]:
display(results["family_review"])
display(Markdown(
    "**Selected status:** " + summary["family_history_status"] +
    "\n\nDIQ175A is not used as a universal family-history predictor because it "
    "is asked only as a reason for perceived risk. DIQ170 is used only as the "
    "combined `medical_or_family_diabetes_risk` proxy and is never described as "
    "genetics or pure family history."
))

,selected_status,candidate_variable,official_wording,assessment,reason,selected_for_model
0,Combined medical/family-risk proxy found,DIQ175A,Why do you think you are at risk for diabetes ...,Ambiguous / not suitable as a universal family...,"Asked only after reported perceived risk; 7,75...",False
1,Combined medical/family-risk proxy found,DIQ170,Ever told by a health professional of health c...,Combined medical/family-risk proxy found,The wording combines medical conditions and fa...,True
2,Combined medical/family-risk proxy found,MCQ_J and other questionnaire files,No direct universal family-history field found...,No direct family-history variable found,MCQ_J is absent and no participant diagnosis w...,False


**Selected status:** Combined medical/family-risk proxy found

DIQ175A is not used as a universal family-history predictor because it is asked only as a reason for perceived risk. DIQ170 is used only as the combined `medical_or_family_diabetes_risk` proxy and is never described as genetics or pure family history.

## 4. Diet engineering

In [5]:
display(Markdown(
    "**Verified diet variables:** " + ", ".join(summary["diet_variables_found"]) +
    "\n\nFor nutrients, the participant value is the mean of day 1 and day 2 "
    "when both exist, day 1 when day 2 is missing, and missing when both are "
    "missing. `diet_recall_days_available` records whether one or two days were used."
))
display(results["diet_unadjusted"].head(20))
display(results["diet_adjusted"].head(20))

**Verified diet variables:** diet_calories, diet_carbohydrate_g, diet_total_sugar_g, diet_fiber_g, diet_protein_g, diet_total_fat_g, diet_saturated_fat_g, diet_sodium_mg, diet_potassium_mg, diet_cholesterol_mg, diet_alcohol_g, meals_away_from_home_7d, fast_food_meals_7d, ready_to_eat_meals_30d, frozen_meals_30d, self_rated_diet_quality

For nutrients, the participant value is the mean of day 1 and day 2 when both exist, day 1 when day 2 is missing, and missing when both are missing. `diet_recall_days_available` records whether one or two days were used.

,variable,participants,effect_scale,odds_ratio,odds_ratio_per_1_sd,odds_ratio_yes_vs_no,ci_lower,ci_upper,p_value,elevated_group_median,non_elevated_group_median,missing_percent,interpretation
0,diet_calories,5523,Per 1 standard deviation,0.929615,0.929615,NaN,0.879999,0.982028,9.107629e-03,1.843000e+03,1.887500e+03,8.635236,"Exploratory association, not a causal effect"
1,diet_carbohydrate_g,5523,Per 1 standard deviation,0.926684,0.926684,NaN,0.877105,0.979065,6.645294e-03,2.184525e+02,2.260850e+02,8.635236,"Exploratory association, not a causal effect"
2,diet_total_sugar_g,5523,Per 1 standard deviation,0.917133,0.917133,NaN,0.867382,0.969739,2.367226e-03,8.556500e+01,9.111000e+01,8.635236,"Exploratory association, not a causal effect"
3,diet_fiber_g,5523,Per 1 standard deviation,1.089463,1.089463,NaN,1.032567,1.149493,1.741877e-03,1.485000e+01,1.385000e+01,8.635236,"Exploratory association, not a causal effect"
4,diet_protein_g,5523,Per 1 standard deviation,0.968627,0.968627,NaN,0.917361,1.022757,2.505945e-01,7.135750e+01,7.090500e+01,8.635236,"Exploratory association, not a causal effect"
5,diet_total_fat_g,5523,Per 1 standard deviation,1.000999,1.000999,NaN,0.948314,1.056611,9.711345e-01,7.432250e+01,7.474500e+01,8.635236,"Exploratory association, not a causal effect"
6,diet_saturated_fat_g,5523,Per 1 standard deviation,0.954107,0.954107,NaN,0.903394,1.007667,9.181694e-02,2.329650e+01,2.394550e+01,8.635236,"Exploratory association, not a causal effect"
7,diet_sodium_mg,5523,Per 1 standard deviation,0.955648,0.955648,NaN,0.904765,1.009392,1.041447e-01,3.012750e+03,3.084500e+03,8.635236,"Exploratory association, not a causal effect"
8,diet_potassium_mg,5523,Per 1 standard deviation,1.055021,1.055021,NaN,0.999745,1.113353,5.109421e-02,2.340250e+03,2.234000e+03,8.635236,"Exploratory association, not a causal effect"
9,diet_cholesterol_mg,5523,Per 1 standard deviation,1.084696,1.084696,NaN,1.028060,1.144451,2.964597e-03,2.585000e+02,2.390000e+02,8.635236,"Exploratory association, not a causal effect"


,variable,participants,effect_scale,odds_ratio,odds_ratio_per_1_sd,odds_ratio_yes_vs_no,ci_lower,ci_upper,p_value,elevated_group_median,non_elevated_group_median,missing_percent,interpretation
0,diet_calories,4729,Per 1 standard deviation,0.974129,0.974129,NaN,0.906734,1.046533,4.736365e-01,1.843000e+03,1.887500e+03,8.635236,"Exploratory association, not a causal effect"
1,diet_carbohydrate_g,4729,Per 1 standard deviation,1.021393,1.021393,NaN,0.953149,1.094523,5.485447e-01,2.184525e+02,2.260850e+02,8.635236,"Exploratory association, not a causal effect"
2,diet_total_sugar_g,4729,Per 1 standard deviation,0.964000,0.964000,NaN,0.901479,1.030857,2.838663e-01,8.556500e+01,9.111000e+01,8.635236,"Exploratory association, not a causal effect"
3,diet_fiber_g,4729,Per 1 standard deviation,1.056831,1.056831,NaN,0.988865,1.129468,1.031423e-01,1.485000e+01,1.385000e+01,8.635236,"Exploratory association, not a causal effect"
4,diet_protein_g,4729,Per 1 standard deviation,0.991549,0.991549,NaN,0.922867,1.065342,8.167520e-01,7.135750e+01,7.090500e+01,8.635236,"Exploratory association, not a causal effect"
5,diet_total_fat_g,4729,Per 1 standard deviation,1.020812,1.020812,NaN,0.952192,1.094377,5.618058e-01,7.432250e+01,7.474500e+01,8.635236,"Exploratory association, not a causal effect"
6,diet_saturated_fat_g,4729,Per 1 standard deviation,0.975512,0.975512,NaN,0.909919,1.045833,4.851032e-01,2.329650e+01,2.394550e+01,8.635236,"Exploratory association, not a causal effect"
7,diet_sodium_mg,4729,Per 1 standard deviation,1.020654,1.020654,NaN,0.950240,1.096285,5.751206e-01,3.012750e+03,3.084500e+03,8.635236,"Exploratory association, not a causal effect"
8,diet_potassium_mg,4729,Per 1 standard deviation,0.952126,0.952126,NaN,0.888572,1.020226,1.639718e-01,2.340250e+03,2.234000e+03,8.635236,"Exploratory association, not a causal effect"
9,diet_cholesterol_mg,4729,Per 1 standard deviation,1.017716,1.017716,NaN,0.949855,1.090425,6.179417e-01,2.585000e+02,2.390000e+02,8.635236,"Exploratory association, not a causal effect"


## 5. Leakage audit

In [6]:
audit = results["audit"]
display(audit)
assert audit["allowed"].all()
print("PASS: no forbidden target or direct-proxy field appears among predictors.")

,model,predictor,matched_forbidden_terms,direct_target_or_proxy,allowed,review_note
0,Model 0: Age only,age,,False,True,Passed name-based audit
1,Model 1: Age + BMI,age,,False,True,Passed name-based audit
2,Model 1: Age + BMI,bmi,,False,True,Passed name-based audit
3,Baseline 3: Age + BMI + systolic BP,age,,False,True,Passed name-based audit
4,Baseline 3: Age + BMI + systolic BP,bmi,,False,True,Passed name-based audit
...,...,...,...,...,...,...
201,Model 14: Full with BMI only,fast_food_meals_7d,,False,True,Passed name-based audit
202,Model 14: Full with BMI only,ready_to_eat_meals_30d,,False,True,Passed name-based audit
203,Model 14: Full with BMI only,frozen_meals_30d,,False,True,Passed name-based audit
204,Model 14: Full with BMI only,self_rated_diet_quality,,False,True,Passed name-based audit


PASS: no forbidden target or direct-proxy field appears among predictors.


## 6. Strict, shared-cohort, and available-case model comparisons

In [7]:
comparison = results["comparison"]
display(results["cohort_report"])
display(comparison.sort_values(["comparison", "roc_auc"], ascending=[True, False]))
display(results["sample_size"])
display(results["integrity_audit"])
assert results["integrity_audit"]["training_only_preprocessing"].all()
assert results["integrity_audit"]["untouched_held_out_test_set"].all()
assert results["integrity_audit"]["no_hba1c_or_diagnosis_leakage"].all()
print("PASS: training-only pipelines, untouched tests, leakage controls, and paired participant checks verified.")

,cohort,participant_count,predictor_scope,minimum_predictor_fraction,required_nonmissing_predictors,notes
0,Strict complete-case,2713,age; bmi; systolic_bp; sleep_hours; resting_he...,1.0,29,Used for the global complete-case comparison.
1,Shared-comparison,2713,sleep_hours; resting_heart_rate; systolic_bp; ...,1.0,29,"Used only for KPI + demographics, diet, proxy,..."
2,Available-case full model,5671,sleep_hours; resting_heart_rate; systolic_bp; ...,0.6,18,Eligibility checked before training-only imput...


,comparison,model,estimator,features,number_of_participants,positive_class_prevalence,training_count,test_count,cv_roc_auc_training_only,best_parameters,...,brier_score,roc_auc_ci_lower,roc_auc_ci_upper,pr_auc_ci_lower,pr_auc_ci_upper,recall_ci_lower,recall_ci_upper,specificity_ci_lower,specificity_ci_upper,bootstrap_resamples
15,Available-case,Model 6: + all verified diet,Regularized logistic regression,sleep_hours; resting_heart_rate; systolic_bp; ...,5696,0.392205,4556,1140,0.800949,"{""model__C"": 1.0}",...,0.182364,0.776462,0.828321,0.636929,0.732593,0.752735,0.828198,0.675244,0.745434,1000
23,Available-case,Model 13: Full without family/proxy,Regularized logistic regression,sleep_hours; resting_heart_rate; systolic_bp; ...,5696,0.392205,4556,1140,0.800949,"{""model__C"": 1.0}",...,0.182364,0.776874,0.829008,0.637635,0.730311,0.753473,0.828644,0.678733,0.743592,1000
5,Available-case,Baseline 3: Age + BMI + systolic BP,Regularized logistic regression,age; bmi; systolic_bp,5966,0.397251,4772,1194,0.788088,"{""model__C"": 1.0}",...,0.180894,0.774614,0.823136,0.651612,0.735515,0.674363,0.756774,0.728511,0.792374,1000
33,Available-case,Model 11: Full verified,Gradient boosting,sleep_hours; resting_heart_rate; systolic_bp; ...,5671,0.389526,4536,1135,0.812078,{},...,0.178540,0.766789,0.819278,0.642482,0.736593,0.604750,0.693880,0.750742,0.813033,1000
13,Available-case,Model 5: + diet behaviours,Regularized logistic regression,sleep_hours; resting_heart_rate; systolic_bp; ...,5809,0.407299,4647,1162,0.788887,"{""model__C"": 1.0}",...,0.184371,0.766665,0.820051,0.651229,0.738101,0.702759,0.778689,0.702243,0.767735,1000
32,Available-case,Model 11: Full verified,Random forest,sleep_hours; resting_heart_rate; systolic_bp; ...,5671,0.389526,4536,1135,0.811961,{},...,0.185386,0.762193,0.814678,0.635522,0.724511,0.685013,0.772730,0.676214,0.743188,1000
25,Available-case,Model 14: Full with BMI only,Regularized logistic regression,sleep_hours; resting_heart_rate; systolic_bp; ...,5672,0.389457,4537,1135,0.805132,"{""model__C"": 1.0}",...,0.187751,0.759430,0.812473,0.606336,0.704663,0.698661,0.785714,0.698322,0.764539,1000
19,Available-case,Model 11: Full verified,Regularized logistic regression,sleep_hours; resting_heart_rate; systolic_bp; ...,5671,0.389526,4536,1135,0.805254,"{""model__C"": 1.0}",...,0.188747,0.759436,0.810151,0.621722,0.709397,0.707560,0.783302,0.686211,0.752959,1000
11,Available-case,Model 4: + diet nutrients,Regularized logistic regression,sleep_hours; resting_heart_rate; systolic_bp; ...,5521,0.390147,4416,1105,0.802799,"{""model__C"": 1.0}",...,0.190428,0.755666,0.809354,0.614887,0.710980,0.686921,0.769741,0.679758,0.745849,1000
9,Available-case,Model 3: Core KPIs + demographics,Regularized logistic regression,sleep_hours; resting_heart_rate; systolic_bp; ...,5474,0.424735,4379,1095,0.788903,"{""model__C"": 1.0}",...,0.195244,0.741597,0.798494,0.624774,0.719271,0.690570,0.771194,0.647252,0.715258,1000


,comparison,model,estimator,starting_participant_count,participants_excluded_target_missing,predictor_availability_fraction_required,required_nonmissing_predictors,participants_excluded_predictor_availability_rule,participants_excluded_all_predictors_missing,participants_excluded_complete_case_rule,final_training_count,final_test_count,positive_outcome_count,negative_outcome_count
0,Complete-case,Model 0: Age only,Regularized logistic regression,6045,0,1.0,1,784,784,3332,2170,543,894,1819
1,Available-case,Model 0: Age only,Regularized logistic regression,6045,0,0.6,1,784,784,0,4208,1053,2310,2951
2,Complete-case,Model 1: Age + BMI,Regularized logistic regression,6045,0,1.0,2,872,8,3332,2170,543,894,1819
3,Available-case,Model 1: Age + BMI,Regularized logistic regression,6045,0,0.6,2,872,8,0,4138,1035,2267,2906
4,Complete-case,Baseline 3: Age + BMI + systolic BP,Regularized logistic regression,6045,0,1.0,3,1100,5,3332,2170,543,894,1819
5,Available-case,Baseline 3: Age + BMI + systolic BP,Regularized logistic regression,6045,0,0.6,2,79,5,0,4772,1194,2370,3596
6,Complete-case,Model 2: Core KPIs,Regularized logistic regression,6045,0,1.0,9,1195,4,3332,2170,543,894,1819
7,Available-case,Model 2: Core KPIs,Regularized logistic regression,6045,0,0.6,6,551,4,0,4395,1099,2325,3169
8,Complete-case,Model 3: Core KPIs + demographics,Regularized logistic regression,6045,0,1.0,12,1788,0,3332,2170,543,894,1819
9,Available-case,Model 3: Core KPIs + demographics,Regularized logistic regression,6045,0,0.6,8,571,0,0,4379,1095,2325,3149


,audit_type,comparison,model,estimator,training_only_preprocessing,untouched_held_out_test_set,no_hba1c_or_diagnosis_leakage,same_participants_for_direct_comparison,comparison_partner,status
0,model,Complete-case,Model 0: Age only,Regularized logistic regression,True,True,True,NaN,,verified
1,model,Available-case,Model 0: Age only,Regularized logistic regression,True,True,True,NaN,,verified
2,model,Complete-case,Model 1: Age + BMI,Regularized logistic regression,True,True,True,NaN,,verified
3,model,Available-case,Model 1: Age + BMI,Regularized logistic regression,True,True,True,NaN,,verified
4,model,Complete-case,Baseline 3: Age + BMI + systolic BP,Regularized logistic regression,True,True,True,NaN,,verified
5,model,Available-case,Baseline 3: Age + BMI + systolic BP,Regularized logistic regression,True,True,True,NaN,,verified
6,model,Complete-case,Model 2: Core KPIs,Regularized logistic regression,True,True,True,NaN,,verified
7,model,Available-case,Model 2: Core KPIs,Regularized logistic regression,True,True,True,NaN,,verified
8,model,Complete-case,Model 3: Core KPIs + demographics,Regularized logistic regression,True,True,True,NaN,,verified
9,model,Available-case,Model 3: Core KPIs + demographics,Regularized logistic regression,True,True,True,NaN,,verified


PASS: training-only pipelines, untouched tests, leakage controls, and paired participant checks verified.


## 7. Incremental predictive value

In [8]:
display(results["incremental"])
display(Markdown(
    "A feature group is not called meaningfully helpful when its gain is tiny "
    "or the paired bootstrap interval includes no improvement."
))

,comparison,comparison_cohort,base_model,expanded_model,held_out_participants,same_test_participants,delta_roc_auc,delta_pr_auc,delta_recall,delta_brier_score,delta_roc_auc_ci_lower,delta_roc_auc_ci_upper,delta_pr_auc_ci_lower,delta_pr_auc_ci_upper,delta_recall_ci_lower,delta_recall_ci_upper,delta_brier_score_ci_lower,delta_brier_score_ci_upper,interpretation
0,Add demographics,Complete-case,Model 2: Core KPIs,Model 3: Core KPIs + demographics,543,True,0.053671,0.068598,0.027933,-0.024164,0.021852,0.086388,0.013597,0.119994,-0.043223,0.095251,-0.035821,-1.273499e-02,Meaningful added predictive information
1,Add all verified diet,Shared-comparison,Model 3: Core KPIs + demographics,Model 6: + all verified diet,543,True,0.004574,0.013202,0.011173,-0.002706,-0.009080,0.017950,-0.008712,0.036739,-0.028417,0.052326,-0.007759,2.287694e-03,No clear meaningful improvement; gain is small...
2,Add medical/family-risk proxy,Shared-comparison,Model 3: Core KPIs + demographics,Model 7P: + medical/family-risk proxy,543,True,0.005326,0.004864,-0.011173,-0.002627,-0.001343,0.012248,-0.008408,0.016516,-0.047632,0.022867,-0.005413,-5.358740e-07,No clear meaningful improvement; gain is small...
3,Add all shared verified groups,Shared-comparison,Model 3: Core KPIs + demographics,Shared full: KPIs + demographics + diet + proxy,543,True,0.009147,0.015663,0.011173,-0.005569,-0.005071,0.024576,-0.008682,0.043117,-0.029078,0.051157,-0.011547,-2.685019e-04,No clear meaningful improvement; gain is small...
4,Full versus age + BMI,Complete-case,Model 1: Age + BMI,Model 11: Full verified,543,True,0.018786,0.025515,0.044693,-0.009546,-0.001781,0.038395,-0.010990,0.063016,-0.005718,0.091990,-0.017428,-1.651071e-03,No clear meaningful improvement; gain is small...
5,Add weight beyond BMI,Complete-case,Model 14: Full with BMI only,Model 11: Full verified,543,True,-0.002410,-0.007925,-0.005587,0.001087,-0.006962,0.002981,-0.017780,0.002780,-0.028257,0.017647,-0.000760,2.959916e-03,No clear meaningful improvement; gain is small...


A feature group is not called meaningfully helpful when its gain is tiny or the paired bootstrap interval includes no improvement.

## 8. Ablation analysis

In [9]:
display(results['ablation'])

,removed_group,status,removed_features,delta_roc_auc,delta_pr_auc,delta_recall,delta_brier_score
0,Diet nutrients,evaluated on same held-out participants,diet_calories; diet_carbohydrate_g; diet_total...,-0.002931,-0.010457,0.000000,0.002339
1,Diet behaviours,evaluated on same held-out participants,meals_away_from_home_7d; fast_food_meals_7d; r...,-0.001120,-0.000611,0.005587,0.000519
2,Family history,not available in full model,,NaN,NaN,NaN,NaN
3,Medical/family-risk proxy,evaluated on same held-out participants,medical_or_family_diabetes_risk,-0.004574,-0.002461,0.000000,0.002863
4,Smoking,not available in full model,,NaN,NaN,NaN,NaN
5,Food security,not available in full model,,NaN,NaN,NaN,NaN
6,Healthcare access,not available in full model,,NaN,NaN,NaN,NaN
7,Demographics,evaluated on same held-out participants,age; sex; income_context,-0.049850,-0.060064,0.000000,0.025147
8,Core KPIs,evaluated on same held-out participants,sleep_hours; resting_heart_rate; systolic_bp; ...,-0.017942,-0.049814,-0.039106,0.006255
9,Activity variables,evaluated on same held-out participants,vigorous_activity; moderate_activity; sedentar...,0.001596,-0.001490,0.000000,-0.000316


## 9. Family-specific analysis

In [10]:
display(results['family_analysis'])

,family_history_status,analysis_label,response,participant_count,elevated_hba1c_prevalence,warning,unadjusted_effect_scale,unadjusted_odds_ratio_yes_vs_no,unadjusted_ci_lower,unadjusted_ci_upper,adjusted_effect_scale,adjusted_odds_ratio_yes_vs_no,adjusted_ci_lower,adjusted_ci_upper,incremental_roc_auc,incremental_interpretation
0,Combined medical/family-risk proxy found,Reported medical or family-history diabetes risk,No,4359,0.295251,This is not genetics or pure family history.,Yes versus No,1.502052,1.290075,1.748859,Yes versus No,1.452188,1.214242,1.736763,0.005326,No clear meaningful improvement; gain is small...
1,Combined medical/family-risk proxy found,Reported medical or family-history diabetes risk,Yes,857,0.386231,This is not genetics or pure family history.,Yes versus No,1.502052,1.290075,1.748859,Yes versus No,1.452188,1.214242,1.736763,0.005326,No clear meaningful improvement; gain is small...


## 10. WellPath survey recommendations

In [11]:
display(results['recommendations'])

,Candidate survey field,NHANES source variable,Dataset evidence available,Incremental model value,Recommendation,Reason,Privacy or sensitivity concern,App usage,Limitation
0,Family history of diabetes,No direct universal variable,No,No clear meaningful improvement; gain is small...,Unavailable,Collect directly if desired; DIQ170 is combined.,Sensitive family medical information,Optional onboarding survey,Not evaluated directly in loaded NHANES files
1,Smoking status,SMQ_J absent,No,Not evaluated,Unavailable,Required source file was absent.,Sensitive health behaviour,Optional onboarding survey,No dataset evidence in this run
2,Fast-food frequency,DBD900,Yes,No clear meaningful improvement; gain is small...,Optional,Available but self-reported.,Low-to-moderate sensitivity,Optional context,Seven-day behaviour may vary
3,Meals away from home,DBD895,Yes,No clear meaningful improvement; gain is small...,Optional,Available but self-reported.,Low-to-moderate sensitivity,Optional context,Seven-day behaviour may vary
4,Food security,FSQ_J absent,No,Not evaluated,Unavailable,Required source file was absent.,High socioeconomic sensitivity,Optional survey only with clear purpose,No dataset evidence in this run
5,Healthcare access,HUQ_J absent,No,Not evaluated,Unavailable,Required source file was absent.,Sensitive access/insurance information,Optional survey,No dataset evidence in this run
6,Income context,INDFMPIR,Yes,Included with demographics,Optional,May add context but can proxy structural inequ...,High financial sensitivity,Avoid unless essential,Do not use punitively
7,Sleep,SLD012,Yes,Included in KPI models,Keep,Existing KPI and feasible to collect.,Health data,Monitoring signal,Self-report is imprecise
8,Activity,PAQ650; PAQ665; PAD680,Yes,Included in KPI models,Keep,Existing KPI group.,Health/behaviour data,Monitoring signal,Self-report is imprecise
9,Blood pressure,BPXSY1-3; BPXDI1-3,Yes,Included in KPI models,Keep,Objective examination KPI.,Health data,Monitoring signal,Single visit is not a diagnosis


## 11. Saved visualizations

In [12]:
chart_files = sorted((PROJECT_ROOT / "outputs" / "charts").glob("[01][0-9]_*.png"))
print(f"{len(chart_files)} expanded figures saved at 300 DPI:")
for chart in chart_files:
    print("-", chart.name)

16 expanded figures saved at 300 DPI:
- 01_roc_auc_by_feature_set.png
- 02_pr_auc_by_feature_set.png
- 03_recall_by_feature_set.png
- 04_brier_by_feature_set.png
- 05_incremental_auc_gain.png
- 06_incremental_recall_gain.png
- 07_ablation_auc_change.png
- 08_ablation_brier_change.png
- 09_full_logistic_exponentiated_coefficients.png
- 10_final_tree_permutation_importance.png
- 11_final_model_calibration.png
- 12_demonstration_confusion_matrix.png
- 13_diet_feature_missingness.png
- 14_family_history_decision.png
- 15_adjusted_diet_associations.png
- 16_model_sample_sizes.png


## 12. Plain-language result

In [13]:
diet_delta = summary["diet_delta_roc_auc"]
proxy_delta = summary["proxy_delta_roc_auc"]
full_delta = summary["full_vs_age_bmi_delta_roc_auc"]
weight_delta = summary["weight_beyond_bmi_delta_roc_auc"]
display(Markdown(f'''
1. **Outcome:** measured HbA1c at or above 5.7% versus below 5.7%; this is not a diabetes diagnosis.
2. **Cohorts:** strict complete-case n={summary["complete_case_sample_size"]:,}; shared-comparison n={summary["shared_comparison_sample_size"]:,}; full-model available-case n={summary["available_case_full_model_sample_size"]:,}, requiring at least {summary["available_case_min_predictor_fraction"]:.0%} of predictors before imputation.
3. **Existing KPIs:** sleep, resting pulse, blood pressure, BMI, weight, activity, and sedentary time.
4. **Diet:** 11 nutrient totals plus five verified diet behaviours were found.
5. **Family history:** no direct universal field was found. DIQ170 remains only a combined medical/family-risk proxy.
6. **Diet value:** ROC AUC changed by {diet_delta:+.3f} (95% CI {summary["diet_delta_roc_auc_ci_lower"]:+.3f} to {summary["diet_delta_roc_auc_ci_upper"]:+.3f}) on the shared test cohort. {summary["diet_incremental_result"]}.
7. **Proxy value:** ROC AUC changed by {proxy_delta:+.3f} (95% CI {summary["proxy_delta_roc_auc_ci_lower"]:+.3f} to {summary["proxy_delta_roc_auc_ci_upper"]:+.3f}). {summary["proxy_incremental_result"]}.
8. **Full versus age + BMI:** ROC AUC changed by {full_delta:+.3f} (95% CI {summary["full_vs_age_bmi_delta_roc_auc_ci_lower"]:+.3f} to {summary["full_vs_age_bmi_delta_roc_auc_ci_upper"]:+.3f}).
9. **Weight beyond BMI:** ROC AUC changed by {weight_delta:+.3f} (95% CI {summary["weight_beyond_bmi_delta_roc_auc_ci_lower"]:+.3f} to {summary["weight_beyond_bmi_delta_roc_auc_ci_upper"]:+.3f}). {summary["weight_beyond_bmi_result"]}.
10. **Smoking, food security, and healthcare access:** not evaluated because SMQ_J, FSQ_J, and HUQ_J were absent.
11. **Best complete-case model:** {summary["best_complete_case_model"]}, ROC AUC {summary["best_complete_case_roc_auc"]:.3f}.
12. **Best available-case model:** {summary["best_available_case_model"]}, ROC AUC {summary["best_available_case_roc_auc"]:.3f}.
13. **Safe claim:** exploratory cross-sectional associations and added predictive information in this dataset.
14. **Not safe to claim:** diagnosis, future diabetes prediction, causation, genetics, or clinical validation.
'''))


1. **Outcome:** measured HbA1c at or above 5.7% versus below 5.7%; this is not a diabetes diagnosis.
2. **Cohorts:** strict complete-case n=2,713; shared-comparison n=2,713; full-model available-case n=5,671, requiring at least 60% of predictors before imputation.
3. **Existing KPIs:** sleep, resting pulse, blood pressure, BMI, weight, activity, and sedentary time.
4. **Diet:** 11 nutrient totals plus five verified diet behaviours were found.
5. **Family history:** no direct universal field was found. DIQ170 remains only a combined medical/family-risk proxy.
6. **Diet value:** ROC AUC changed by +0.005 (95% CI -0.009 to +0.018) on the shared test cohort. No clear meaningful improvement; gain is small and/or uncertainty includes no improvement.
7. **Proxy value:** ROC AUC changed by +0.005 (95% CI -0.001 to +0.012). No clear meaningful improvement; gain is small and/or uncertainty includes no improvement.
8. **Full versus age + BMI:** ROC AUC changed by +0.019 (95% CI -0.002 to +0.038).
9. **Weight beyond BMI:** ROC AUC changed by -0.002 (95% CI -0.007 to +0.003). No clear meaningful improvement; gain is small and/or uncertainty includes no improvement.
10. **Smoking, food security, and healthcare access:** not evaluated because SMQ_J, FSQ_J, and HUQ_J were absent.
11. **Best complete-case model:** Model 14: Full with BMI only, ROC AUC 0.783.
12. **Best available-case model:** Model 6: + all verified diet, ROC AUC 0.804.
13. **Safe claim:** exploratory cross-sectional associations and added predictive information in this dataset.
14. **Not safe to claim:** diagnosis, future diabetes prediction, causation, genetics, or clinical validation.


## Reproducibility checklist

- All available XPT/CSV files were inspected.
- `SEQN` remains unique after joins.
- Day-one/day-two diet totals use the documented averaging rule.
- Family-history wording and skip patterns were reviewed.
- Leakage audit passed before training.
- Preprocessing remained inside training-fitted pipelines.
- Direct complete-case comparisons share test participants.
- 1,000-resample confidence intervals were generated.
- Tables, models, metadata, and separate 300-DPI charts were saved.
- The WellPath React/Vite applications were not used or modified.

**Research-use warning:** Associations are not causal effects. Optional survey
fields can be sensitive. This prototype is not clinically validated and must not
be used for diagnosis or medical decisions.
- The available-case 60% predictor-availability rule is reported before imputation.
- A dedicated shared cohort excludes unrelated optional groups.
- Medical/family-risk odds ratios use Yes versus No without standardizing the binary predictor.
- Weight beyond BMI is evaluated on identical held-out participants.
- `model_integrity_audit.csv` verifies training-only pipelines, disjoint tests,
  leakage controls, and participant matching for every direct comparison.